# Token Residuals — Fixed Sequence, Evolving Weights

The input sequence never changes across the 35 layers — token at position 325 is always the same token.
What changes is *how much residual weight* each position carries as the representation flows deeper.

This notebook asks: **for each fixed token in the sequence, how does its residual contribution
evolve layer by layer through the sliding-attention stack?**

Two patterns to watch for:
- **Early-dominant**: position contributes heavily in layers 0–10, fades out by layer 25
- **Late-dominant**: position is quiet early, then ramps up in layers 20–34

And the question behind both: is the deep-layer residual routing driven by **semantic content tokens**
or by **structural/syntactic tokens** (punctuation, turn markers, whitespace)?

In [26]:
import pandas as pd
import numpy as np
from pathlib import Path

OUT = Path("outputs")
parquet = OUT / "full" / "cases.parquet"
if not parquet.exists():
    parquet = OUT / "smoke" / "cases.parquet"

df = pd.read_parquet(parquet)
print(f"Source: {parquet}")
print(f"Rows: {len(df):,}  |  Samples: {df['sample_id'].nunique()}  |  Layers: {df['layer'].nunique()}")

# isolate sliding layers — BOS is absent here, so any residual comes from in-window tokens
sliding = df[df["layer_type"] == "sliding_attention"].copy()

# the sequence is fixed: same key_pos → same token across all layers within a sample
# verify this holds
assert sliding.groupby(["sample_id", "key_pos"])["decoded_token"].nunique().max() == 1, \
    "same position has different tokens — unexpected"
print("\nVerified: token at each position is identical across all layers (sequence is fixed).")

Source: outputs/smoke/cases.parquet
Rows: 4,891  |  Samples: 5  |  Layers: 35

Verified: token at each position is identical across all layers (sequence is fixed).


## 1. Build the position × layer residual matrix

For each sample, pivot the data into a matrix where:
- rows = key positions (each a fixed token in the sequence)
- columns = sliding layer index
- values = mean `score_pre` across all anchor queries that saw this (position, layer) pair

This lets us read off each token's "residual profile" across depth.

In [27]:
SLIDING_LAYERS = sorted(sliding["layer"].unique())

def build_profile(sample_id):
    """Returns (pivot_df, token_map) for one sample.
    pivot_df: index=key_pos, columns=layer, values=mean score_pre
    token_map: {key_pos: decoded_token}
    """
    s = sliding[sliding["sample_id"] == sample_id]
    pivot = (
        s.groupby(["key_pos", "layer"])["score_pre"]
        .mean()
        .unstack(fill_value=np.nan)
    )
    pivot = pivot.reindex(columns=SLIDING_LAYERS)  # consistent column order
    token_map = s.groupby("key_pos")["decoded_token"].first()
    pivot.index = [f"{pos:4d}  {repr(token_map[pos]):22s}" for pos in pivot.index]
    return pivot, token_map

sample_ids = sorted(sliding["sample_id"].unique())
print(f"Samples available: {sample_ids}")
print(f"Sliding layers: {SLIDING_LAYERS}")

Samples available: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)]
Sliding layers: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(20), np.int64(21), np.int64(22), np.int64(23), np.int64(25), np.int64(26), np.int64(27), np.int64(28), np.int64(30), np.int64(31), np.int64(32), np.int64(33)]


## 2. Top positions — which tokens carry the most residual weight?

For each sample, rank key positions by their **mean score_pre across all sliding layers**.
This surfaces tokens that are persistently high throughout the stack.

In [28]:
for sid in sample_ids:
    s = sliding[sliding["sample_id"] == sid]
    stratum = s["stratum"].iloc[0]
    token_map = s.groupby("key_pos")["decoded_token"].first()

    mean_by_pos = s.groupby("key_pos")["score_pre"].mean().sort_values(ascending=False)
    print(f"\n=== Sample {sid}  ({stratum}) ===")
    print(f"{'pos':>6}  {'token':25s}  {'mean score_pre':>14}  {'#layers seen':>12}")
    n_layers_seen = s.groupby("key_pos")["layer"].nunique()
    for pos, score in mean_by_pos.head(15).items():
        tok = repr(token_map[pos])
        nlayers = n_layers_seen.get(pos, 0)
        print(f"  {pos:4d}  {tok:25s}  {score:14.2f}  {nlayers:12d}")


=== Sample 0  (chat/en) ===
   pos  token                      mean score_pre  #layers seen
   511  ' Quick'                            10.58            28
   541  ' the'                               9.02             1
   574  ' for'                               8.70            11
   243  '-'                                  7.33            21
   359  "'"                                  6.91            21
   498  '<|turn>'                            6.43             1
   331  'Show'                               6.16             5
   553  ' the'                               6.00             5
   499  'user'                               5.90             3
   567  ' To'                                5.72             9
   638  " '"                                 5.53            22
   512  ' Shop'                              5.47            28
   447  'ify'                                5.28             2
   570  ' this'                              5.13             7
   572  ' i

## 3. Early-dominant vs late-dominant tokens

Split sliding layers into depth bands and compare which token positions dominate in each band.
A token that is top-3 in early layers but absent in late layers tells a different story
than one that ramps up only at depth.

In [29]:
BANDS = {
    "early  (L 0-4) ": list(range(0, 5)),
    "mid    (L 5-14)": list(range(5, 15)),
    "deep   (L15-24)": list(range(15, 25)),
    "latest (L25-34)": list(range(25, 35)),
}

for sid in sample_ids:
    s = sliding[sliding["sample_id"] == sid]
    stratum = s["stratum"].iloc[0]
    token_map = s.groupby("key_pos")["decoded_token"].first()
    print(f"\n=== Sample {sid}  ({stratum}) — top 5 positions per depth band ===")

    for band_name, layers in BANDS.items():
        band_rows = s[s["layer"].isin(layers)]
        if band_rows.empty:
            continue
        top = band_rows.groupby("key_pos")["score_pre"].mean().sort_values(ascending=False).head(5)
        pairs = [(pos, repr(token_map[pos]), f"{sc:.1f}") for pos, sc in top.items()]
        formatted = "  |  ".join(f"{p[0]:4d} {p[1]:18s} ({p[2]})" for p in pairs)
        print(f"  {band_name}:  {formatted}")


=== Sample 0  (chat/en) — top 5 positions per depth band ===
  early  (L 0-4) :   511 ' Quick'           (36.5)  |   553 ' the'             (21.4)  |   574 ' for'             (18.4)  |   638 " '"               (12.7)  |   331 'Show'             (10.3)
  mid    (L 5-14):   511 ' Quick'           (14.6)  |   568 ' check'           (7.6)  |   512 ' Shop'            (7.5)  |   498 '<|turn>'          (6.4)  |   447 'ify'              (6.3)
  deep   (L15-24):   639 'Show'             (6.7)  |   512 ' Shop'            (6.4)  |   511 ' Quick'           (5.6)  |   568 ' check'           (4.0)  |   243 '-'                (3.7)
  latest (L25-34):   359 "'"                (12.9)  |   243 '-'                (11.6)  |   602 ' the'             (7.1)  |   512 ' Shop'            (4.2)  |   511 ' Quick'           (3.7)

=== Sample 1  (named_entity_recognition/en) — top 5 positions per depth band ===
  early  (L 0-4) :   637 ' ("'              (21.5)  |   638 'split'            (21.0)  |   574 'ry'     

## 4. Per-position layer profile for the top tokens

For the top-N positions in each sample, show their score_pre at every sliding layer.
This is the full trajectory — easy to see if a token decays, is stable, or ramps up.

In [30]:
TOP_N = 10  # positions per sample

for sid in sample_ids:
    s = sliding[sliding["sample_id"] == sid]
    stratum = s["stratum"].iloc[0]
    token_map = s.groupby("key_pos")["decoded_token"].first()

    top_pos = s.groupby("key_pos")["score_pre"].mean().nlargest(TOP_N).index.tolist()

    # per-layer mean score_pre for these positions
    sub = s[s["key_pos"].isin(top_pos)]
    pivot = sub.groupby(["key_pos", "layer"])["score_pre"].mean().unstack(fill_value=np.nan)
    pivot = pivot.reindex(columns=SLIDING_LAYERS)

    # label rows with token
    pivot.index = [f"{pos:4d} {repr(token_map[pos]):20s}" for pos in pivot.index]

    # round for readability
    display_piv = pivot.round(1).fillna("-")

    print(f"\n=== Sample {sid} ({stratum}) — score_pre per sliding layer ===")
    print(f"{'position / token':25s}  " + "  ".join(f"L{l:02d}" for l in SLIDING_LAYERS))
    for row_label, row in display_piv.iterrows():
        vals = "  ".join(f"{str(v):4s}" for v in row)
        print(f"  {row_label:25s}  {vals}")


=== Sample 0 (chat/en) — score_pre per sliding layer ===
position / token           L00  L01  L02  L03  L05  L06  L07  L08  L10  L11  L12  L13  L15  L16  L17  L18  L20  L21  L22  L23  L25  L26  L27  L28  L30  L31  L32  L33
   243 '-'                   -     -     -     -     -     -     -     2.5   2.3   2.5   2.8   5.9   2.8   2.7   2.0   3.0   5.2   4.0   5.5   3.4   2.6   8.3   17.8  11.0  12.8  19.9  10.8  9.8 
   331 'Show'                -     13.6  -     6.9   -     -     -     -     -     1.9   1.8   6.6   -     -     -     -     -     -     -     -     -     -     -     -     -     -     -     -   
   359 "'"                   -     -     -     -     -     -     -     1.3   2.2   1.8   2.1   3.2   2.7   1.9   1.4   2.1   4.4   3.3   6.9   3.5   3.5   10.9  16.3  12.2  14.3  20.4  14.4  11.2
   498 '<|turn>'             -     -     -     -     6.4   -     -     -     -     -     -     -     -     -     -     -     -     -     -     -     -     -     -     -     -     -     -  

## 5. Cross-sample: which tokens appear in top positions across all samples?

Pool all samples. For each (decoded_token, position_group), compute mean and count.
Tokens that appear across multiple different prompts at high residual weight are structurally important,
not just artifacts of one prompt's phrasing.

In [31]:
# use score_resid_exact here — ground truth residual shift
# groupby the full dataframe (no column pre-selection) so sample_id is available
cross = (
    sliding
    .groupby(["position_group", "decoded_token"])
    .agg(
        mean_pre=("score_pre", "mean"),
        mean_exact=("score_resid_exact", "mean"),
        appearances=("score_pre", "count"),
        samples_seen=("sample_id", "nunique"),
    )
    .reset_index()
    .sort_values("mean_pre", ascending=False)
)

# show top tokens per group that appear in at least 5 rows
for grp in ["middle", "recent", "self"]:
    sub = cross[(cross["position_group"] == grp) & (cross["appearances"] >= 5)]
    print(f"\n=== {grp} — top tokens (≥5 appearances across all prompts) ===")
    print(sub.head(20).to_string(index=False))


=== middle — top tokens (≥5 appearances across all prompts) ===
position_group decoded_token  mean_pre  mean_exact  appearances  samples_seen
        middle            as 12.251458    7.088907           62             2
        middle                8.704460    3.937957          115             3
        middle             '  8.562211    3.553643          122             2
        middle          Show  6.158481    6.485027            5             1
        middle             -  5.535221    2.407719          159             3
        middle       limited  5.314997    5.488526           10             1
        middle         model  4.553820    7.580691           97             4
        middle             "  4.397517    1.972658          166             2
        middle             .  3.666552    4.471032           31             4
        middle          yuan  3.556963    4.186505            8             1
        middle       Shopify  3.473146    3.943168            9             1

## 6. Do deep layers weight content vs structural tokens differently?

Classify each token as structural (punctuation, whitespace, special markers) or content (actual words).
Compare how each class's share of total score_pre changes from early to late layers.

In [32]:
import re

_structural = re.compile(
    r'^[\s\.,;:!?\(\)\[\]\{}\'\"\`/\\\-–—\|<>\*#@~^_=+]+$|^<[^>]+>$'
)

def classify(tok):
    t = str(tok).strip()
    return "structural" if (not t or _structural.match(t)) else "content"

sliding["token_class"] = sliding["decoded_token"].apply(classify)

print("Score_pre share by token_class and depth band (all samples, all position groups)\n")
print(f"{'band':20s}  {'content share':>14}  {'structural share':>16}  {'content mean_pre':>16}  {'structural mean_pre':>19}")

for band_name, layers in BANDS.items():
    band_rows = sliding[sliding["layer"].isin(layers)]
    total = band_rows["score_pre"].sum()
    by_class = band_rows.groupby("token_class")["score_pre"].agg(["sum", "mean"])
    c_share = by_class.loc["content", "sum"] / total if "content" in by_class.index else 0
    s_share = by_class.loc["structural", "sum"] / total if "structural" in by_class.index else 0
    c_mean  = by_class.loc["content", "mean"] if "content" in by_class.index else 0
    s_mean  = by_class.loc["structural", "mean"] if "structural" in by_class.index else 0
    print(f"  {band_name:18s}  {c_share:14.1%}  {s_share:16.1%}  {c_mean:16.2f}  {s_mean:19.2f}")

Score_pre share by token_class and depth band (all samples, all position groups)

band                   content share  structural share  content mean_pre  structural mean_pre
  early  (L 0-4)               61.6%             38.4%              6.40                 6.32
  mid    (L 5-14)              59.5%             40.5%              3.47                 3.20
  deep   (L15-24)              52.1%             47.9%              2.62                 2.92
  latest (L25-34)              31.4%             68.6%              2.52                 5.59


In [33]:
import re

_structural = re.compile(
    r'^[\s\.,;:!?\(\)\[\]\{\}\'"\`/\\\-–—\|<>\*#@~^_=+]+$'
    r'|^<[^>]+>$'  # special tokens like <|turn|>
)

def classify(tok):
    t = str(tok).strip()
    if not t:
        return "structural"
    return "structural" if _structural.match(t) else "content"

sliding["token_class"] = sliding["decoded_token"].apply(classify)

print("Score_pre share by token_class and depth band (all samples, all position groups)\n")
print(f"{'band':20s}  {'content share':>14}  {'structural share':>16}  {'content mean_pre':>16}  {'structural mean_pre':>19}")

for band_name, layers in BANDS.items():
    band_rows = sliding[sliding["layer"].isin(layers)]
    total = band_rows["score_pre"].sum()
    by_class = band_rows.groupby("token_class")["score_pre"].agg(["sum", "mean"])
    c_share = by_class.loc["content", "sum"] / total if "content" in by_class.index else 0
    s_share = by_class.loc["structural", "sum"] / total if "structural" in by_class.index else 0
    c_mean  = by_class.loc["content", "mean"] if "content" in by_class.index else 0
    s_mean  = by_class.loc["structural", "mean"] if "structural" in by_class.index else 0
    print(f"  {band_name:18s}  {c_share:14.1%}  {s_share:16.1%}  {c_mean:16.2f}  {s_mean:19.2f}")

Score_pre share by token_class and depth band (all samples, all position groups)

band                   content share  structural share  content mean_pre  structural mean_pre
  early  (L 0-4)               61.6%             38.4%              6.40                 6.32
  mid    (L 5-14)              59.5%             40.5%              3.47                 3.20
  deep   (L15-24)              52.1%             47.9%              2.62                 2.92
  latest (L25-34)              31.4%             68.6%              2.52                 5.59


In [34]:
import json

SAVE = OUT / "smoke" / "token_residuals"
SAVE.mkdir(exist_ok=True)

# top positions per sample
rows = []
for sid in sample_ids:
    s = sliding[sliding["sample_id"] == sid]
    token_map   = s.groupby("key_pos")["decoded_token"].first()
    mean_by_pos = s.groupby("key_pos")["score_pre"].mean()
    n_layers    = s.groupby("key_pos")["layer"].nunique()
    stratum     = s["stratum"].iloc[0]
    for pos, score in mean_by_pos.sort_values(ascending=False).head(15).items():
        rows.append({"sample_id": int(sid), "stratum": stratum, "key_pos": int(pos),
                     "token": token_map[pos], "mean_score_pre": round(float(score), 3),
                     "layers_seen": int(n_layers[pos])})
pd.DataFrame(rows).to_csv(SAVE / "top_positions_per_sample.csv", index=False)

# depth-band top-5 per sample
rows = []
for sid in sample_ids:
    s = sliding[sliding["sample_id"] == sid]
    token_map = s.groupby("key_pos")["decoded_token"].first()
    stratum   = s["stratum"].iloc[0]
    for band, layers in BANDS.items():
        band_rows = s[s["layer"].isin(layers)]
        if band_rows.empty: continue
        top = band_rows.groupby("key_pos")["score_pre"].mean().sort_values(ascending=False).head(5)
        for rank, (pos, sc) in enumerate(top.items(), 1):
            rows.append({"sample_id": int(sid), "stratum": stratum, "band": band,
                         "rank": rank, "key_pos": int(pos), "token": token_map[pos],
                         "mean_score_pre": round(float(sc), 3)})
pd.DataFrame(rows).to_csv(SAVE / "band_top_positions.csv", index=False)

# full layer profiles for top-10 positions per sample
rows = []
for sid in sample_ids:
    s = sliding[sliding["sample_id"] == sid]
    token_map = s.groupby("key_pos")["decoded_token"].first()
    stratum   = s["stratum"].iloc[0]
    top_pos   = s.groupby("key_pos")["score_pre"].mean().nlargest(10).index
    sub = s[s["key_pos"].isin(top_pos)]
    pivot = sub.groupby(["key_pos","layer"])["score_pre"].mean().unstack(fill_value=np.nan).reindex(columns=SLIDING_LAYERS)
    for pos in pivot.index:
        for layer in SLIDING_LAYERS:
            val = pivot.loc[pos, layer]
            rows.append({"sample_id": int(sid), "stratum": stratum, "key_pos": int(pos),
                         "token": token_map[pos], "layer": int(layer),
                         "score_pre": None if np.isnan(val) else round(float(val), 3)})
pd.DataFrame(rows).to_csv(SAVE / "position_layer_profiles.csv", index=False)

# cross-sample token summary
cross.to_csv(SAVE / "cross_sample_tokens.csv", index=False)

# structural vs content by band
rows = []
for band, layers in BANDS.items():
    band_rows = sliding[sliding["layer"].isin(layers)]
    total = band_rows["score_pre"].sum()
    by_class = band_rows.groupby("token_class")["score_pre"].agg(["sum", "mean"])
    for cls in ["content", "structural"]:
        if cls in by_class.index:
            rows.append({"band": band, "token_class": cls,
                         "share": round(float(by_class.loc[cls, "sum"] / total), 4),
                         "mean_score_pre": round(float(by_class.loc[cls, "mean"]), 3)})
pd.DataFrame(rows).to_csv(SAVE / "structural_vs_content_by_band.csv", index=False)

print(f"Saved 5 tables to {SAVE}/")

Saved 5 tables to outputs/smoke/token_residuals/
